# 02. Preparing HUXt Inputs, One Operation At A Time

**What this notebook is.** HUXt is a fast solar-wind model: given a map of the wind speed at an inner boundary (a few tens of solar radii) plus a Cone-CME description, it advects that wind out to 1 AU and tells you when a CME reaches Earth. But HUXt cannot start from nothing - it needs a *physically grounded* boundary condition. Building that boundary, for a specific real CME event, is the job of `scripts/generate_huxt_input.py`.

**What we do here.** We take that script apart **one operation at a time**. Each task below reimplements a single step inline on local data, names the script function it mirrors, and explains *why* the step exists in the chain. The end product is a prepared event directory `data_dir/sw/<event>/` containing a HUXt boundary file and a seed-parameter config.

**The physical chain**, from the Sun's surface to a HUXt-ready input:

```text
photospheric magnetic field   (GONG magnetogram)
    -> coronal + solar-wind speed map   (WSA+)
    -> speed Earth actually sees         (sub-Earth track sampling)
    -> 1-D inner boundary for HUXt       (v_boundary_<event>.npz)
    -> + CME launch time and geometry    (event_config.yaml seed)
```

**Cost.** Two steps are expensive: the GONG download needs the network (kept guarded), and the WSA+ map needs the `wsaplus.pt` checkpoint. WSA+ **runs live** here (with a cached fallback); everything else runs live on the local files.

Task order follows the script's own call chain:

1. **`load_events`** - read the seed CME parameters from `events.csv`.
2. **`download_gong_mag`** - fetch GONG magnetograms near onset (guarded).
3. **`find_closest_map`** - pick the magnetogram closest to onset and read its Carrington rotation.
4. **`run_wsaplus`** - build the longitude-latitude WSA+ speed map (live).
5. **`compute_subearth_track`** - trace Earth through the rotation.
6. **`sample_interpolated` / `sample_nearest`** - sample the map along that track.
7. **`map_input_huxt`** - reduce the track to the 1-degree HUXt boundary.
8. **injection time + `create_config`** - compute `inject_hour` and write the seed config.

In [ ]:
# --- Google Colab bootstrap (no-op locally) ---
import sys, os

if "google.colab" in sys.modules:
    REPO_URL = os.environ.get("CONECAST_REPO", "https://github.com/georgemilosh/conecast")
    REPO_DIR = "/content/conecast"
    fresh = not os.path.isdir(REPO_DIR)
    if fresh:
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
        # HUXt + WSA+ (and a current sunpy) are not preinstalled in Colab.
        os.system("pip install -q "
                  "'huxt @ git+https://github.com/University-of-Reading-Space-Science/HUXt' "
                  "wsaplus sunpy")
    os.chdir(REPO_DIR)
    if fresh:
        # Those installs pull a newer NumPy/SciPy; restart so they load cleanly.
        print("Dependencies installed - restarting the Colab runtime.")
        print("When it reconnects, RUN THIS CELL AGAIN (or Runtime > Run all).")
        os.kill(os.getpid(), 9)
    # The WSA+ checkpoint (~317 MB) is fetched from Zenodo on demand by notebook 02.
    print("Colab bootstrap complete; cwd =", os.getcwd())
else:
    print("Not in Colab - using the local checkout.")

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "scripts").exists():
    BASE_DIR = cwd
elif (cwd.parent / "scripts").exists():
    BASE_DIR = cwd.parent
else:
    # Fallback: assume the notebook is run from inside the repo.
    BASE_DIR = cwd

SCRIPT_DIR = BASE_DIR / "scripts"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

print("BASE_DIR =", BASE_DIR)

## Pipeline Overview

`generate_huxt_input.py` is one call chain, run once per selected event:

```text
main()
  ├─ load_events()
  └─ process_event()                  one call per event
       ├─ prepare_background()
       │    ├─ download_gong_mag()         optional, needs network
       │    ├─ find_closest_map()
       │    ├─ run_wsaplus()               skipped if WSA+ cache exists
       │    └─ map_input_huxt()
       │         ├─ compute_subearth_track()
       │         ├─ sample_interpolated()
       │         └─ sample_nearest()
       ├─ rhf.run_huxt_sim()               optional seed sanity run
       └─ write_event_config()             writes event_config.yaml
```

The five Cone-CME parameters carried through to the seed config are:

| index | name | units | meaning |
| --- | --- | --- | --- |
| 0 | `inject_hour` | h | launch time after model start |
| 1 | `longitude` | deg | central longitude |
| 2 | `latitude` | deg | central latitude |
| 3 | `width` | deg | angular half-width |
| 4 | `v` (`speed`) | km/s | initial CME speed |

Only `inject_hour` is computed (from the magnetogram time and `cme_0p1_au`); the other four come straight from the CSV.

> **Why a separate "background" step?** A CME does not travel through a vacuum - it ploughs into the ambient solar wind, which can be fast or slow depending on where you look. Getting the *arrival time* right means getting that background right first. Tasks 2-7 build the background; Task 8 adds the CME on top.

## Setup

Import the shared libraries and choose one event. `event_dir` is the per-event directory the script fills in; every task re-loads what it needs from it, so the tasks can be run independently once this cell has executed.

> **Prepared data required (Tasks 3-9).** This repo ships only source + the seed config; the GONG magnetograms, WSA+ map, and boundary are *not* included. Tasks 1-2 run anywhere, but Tasks 3 onward need prepared inputs for the chosen event. Generate them once with `python scripts/generate_huxt_input.py --event <event>` (downloads GONG + runs WSA+; needs `data_dir/sw/wsaplus.pt`). Until then, Task 3 stops with a clear message.

> **Tip.** Change `event` to any prepared event to re-run the whole walkthrough for a different CME.

In [ ]:
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from pathlib import Path
import astropy.units as u
from astropy.time import Time
from IPython.display import display

DATA_ROOT = BASE_DIR / "data_dir" / "sw"
EVENTS_CSV = BASE_DIR / "data_dir" / "events.csv"

event = "2017-09-06"
event_dir = DATA_ROOT / event
print("event:", event, "| dir exists:", event_dir.exists())
for path in sorted(event_dir.glob("*"))[:10]:
    print("  ", path.name)

## Task 1: Read The Event Catalogue

*Mirrors `load_events()` and `truthy()`.*

Everything starts from `data_dir/events.csv`, a small catalogue where each row is one observed CME with the numbers needed to seed a simulation. These come from observations (coronagraph fits, CME catalogues) - they are the *first guess* that the downstream GP surrogate work then refines.

**Columns the script requires:**

| column | meaning |
| --- | --- |
| `event` | event label, also the output directory name |
| `cme_onset` | time the CME is first seen (used to pick the magnetogram) |
| `cme_0p1_au` | time the CME front reaches 0.1 AU (sets the launch time) |
| `longitude`, `latitude` | CME nose direction in degrees |
| `width` | angular half-width of the cone |
| `speed` | initial radial speed (km/s) |

An optional `enabled` column lets you switch rows off without deleting them; `truthy()` decides what counts as "on". This task reads the CSV, checks the required columns exist, drops disabled rows, and pulls out one event.

> **What to look for:** the selected row's `speed` and `width` are the CME's headline properties; `cme_onset` and `cme_0p1_au` are a few hours apart and together pin down *when* to launch.

In [ ]:
REQUIRED = {"event", "cme_onset", "cme_0p1_au", "longitude", "latitude", "width", "speed"}

def truthy(value):
    return str(value).strip().lower() in {"1", "true", "yes", "y", "on"}

events_table = pd.read_csv(EVENTS_CSV)
print("missing required columns:", (REQUIRED - set(events_table.columns)) or "none")

if "enabled" in events_table.columns:
    keep = events_table["enabled"].map(lambda v: str(v).strip() == "" or truthy(v))
    events_table = events_table[keep]
display(events_table.head())

selected_row = events_table.loc[events_table["event"] == event].iloc[0]
print("seed row for", event, ":")
display(selected_row[["event", "cme_onset", "cme_0p1_au", "longitude", "latitude", "width", "speed"]])

## Task 2: Download The GONG Magnetogram (guarded)

*Mirrors `download_gong_mag()`.*

**What GONG is.** The Global Oscillation Network Group operates ground-based solar telescopes that, among other things, publish *synoptic magnetograms*: full-Sun maps of the photospheric line-of-sight magnetic field in Carrington coordinates, updated roughly hourly. That surface field is the boundary condition for any coronal model - it is where the solar wind ultimately comes from.

**Why near the onset.** The corona evolves, so we want the magnetic map that best represents the Sun *at the moment the CME launched*. The script searches a 6-hour window ending at `cme_onset` and takes what GONG has there.

**How the fetch works.** SunPy's `Fido` is a federated search/download client: `Fido.search(Time(...), Instrument("GONG"))` finds matching files and `Fido.fetch(...)` downloads them. GONG files arrive gzip-compressed (`.fits.gz`); Task 3 decompresses them to `.fits` before reading.

> **Flaky server.** `gong2.nso.edu` sometimes drops connections (SSL / connect errors) and only part of a batch downloads. That is fine - Task 3 just picks the closest of whatever arrived, so one good map is enough. Re-run this cell to retry the rest.

This needs network access, so the live fetch is guarded by `RUN_DOWNLOAD`. We instead list the FITS already present - exactly what a real run would have left behind.

> **Side note.** "Synoptic" means the map is assembled over a full rotation, so a single magnetogram already spans all 360 deg of Carrington longitude - that is what makes it usable as a global boundary later.

In [ ]:
from sunpy.net import Fido, attrs as a

t_cme = Time(selected_row["cme_onset"], scale="utc")
t_start_mag = t_cme - 6 * u.hour
t_end_mag = t_cme
print("CME onset:", t_cme.isot)
print("GONG search window:", t_start_mag.isot, "->", t_end_mag.isot)

local_fits = sorted(event_dir.glob("*.fits"))
print("local GONG FITS:", [p.name for p in local_fits])

RUN_DOWNLOAD = False   # set True to query/fetch over the network
if RUN_DOWNLOAD:
    res = Fido.search(a.Time(t_start_mag, t_end_mag), a.Instrument("GONG"))
    print(res)
    files = Fido.fetch(res, path=str(event_dir / "{file}"))
    print("downloaded:", files)
elif local_fits:
    print("Skipping live download; reusing the local FITS above.")
else:
    print("No local GONG FITS found. The data tasks (3-9) need prepared inputs.")
    print("Generate them with (downloads GONG + runs WSA+; needs data_dir/sw/wsaplus.pt):")
    print(f"  python scripts/generate_huxt_input.py --event {event}")
    print("...or set RUN_DOWNLOAD = True above to fetch just the magnetograms here.")

## Task 3: Select The Closest Magnetogram

*Mirrors `find_closest_map()`.*

The download window may have left several FITS files. This step loads each one as a SunPy `Map`, reads its observation time, and keeps the one with the smallest `|obs_time - cme_onset|`.

**Carrington rotation number.** From the chosen map's date we read its *Carrington rotation* (CR) number - a running count of solar rotations (~27.27 days each) used as the Sun's natural calendar. The CR number, a non-integer here, encodes both which rotation and how far through it we are; Task 5 uses it to work out where Earth sits in Carrington longitude.

**Reading the magnetogram.** The plot uses a symmetric color range (`vmin = -vmax`) so that zero field is white, red is one magnetic polarity and blue the other. The large bipolar regions are active regions / sunspot groups; the quiet background is weak mixed field.

> **What to look for:** the printed `dt[hr]` should be small (a fraction of an hour to a few hours) - if it is large, GONG had a gap near onset and the background is less trustworthy.

In [ ]:
import sunpy.map
from sunpy.coordinates.sun import carrington_rotation_number
from astropy.io import fits

# GONG magnetograms arrive gzipped; decompress any *.fits.gz so they are picked up
# (mirrors download_gong_mag() in generate_huxt_input.py).
for gzfile in sorted(event_dir.glob("*.fits.gz")):
    fitsfile = gzfile.with_suffix("")  # strip .gz -> .fits
    with fits.open(gzfile) as hdul:
        hdul.writeto(fitsfile, overwrite=True)
    gzfile.unlink()
    print("decompressed:", fitsfile.name)

rows = []
for fits_path in sorted(event_dir.glob("*.fits")):
    try:
        gong_map = sunpy.map.Map(fits_path)
        rows.append({
            "file": fits_path.name,
            "obs_time": gong_map.date.isot,
            "dt_hours": abs((gong_map.date - t_cme).to(u.hour).value),
        })
    except Exception as exc:
        print("skip", fits_path.name, "|", exc)

if not rows:
    raise FileNotFoundError(
        f"No GONG FITS files in {event_dir}.\n"
        f"Tasks 3-9 need prepared inputs for this event. Generate them first with:\n"
        f"  python scripts/generate_huxt_input.py --event {event}\n"
        f"(downloads GONG magnetograms + runs WSA+; needs data_dir/sw/wsaplus.pt)."
    )

magnetograms = pd.DataFrame(rows).sort_values("dt_hours").reset_index(drop=True)
display(magnetograms)

closest_file = event_dir / magnetograms.iloc[0]["file"]
closest_map = sunpy.map.Map(closest_file)
closest_time = closest_map.date
cr_num = float(carrington_rotation_number(closest_time))
print("closest:", closest_file.name, "| dt[hr]:", magnetograms.iloc[0]["dt_hours"], "| CR:", cr_num)

lim = float(np.nanmax(np.abs(closest_map.data)))
plt.figure(figsize=(8, 4))
closest_map.plot(cmap="RdBu_r", vmin=-lim, vmax=lim)
plt.colorbar(label="B [G]")
plt.title(f"Closest GONG magnetogram: {closest_file.name}")
plt.tight_layout()

## Task 4: Build The WSA+ Speed Map (live)

*Mirrors `run_wsaplus()`.*

**What WSA+ does.** WSA (Wang-Sheeley-Arge) is the classic empirical recipe for turning a photospheric magnetogram into a solar-wind *speed* map: trace the coronal magnetic field (a potential-field source-surface extrapolation), measure how fast flux tubes expand and how far each footpoint sits from the nearest coronal-hole boundary, and feed those two geometric quantities into an empirical speed formula. Open-field regions (coronal holes) give fast wind; field near the streamer belt gives slow wind. The "+" here is a machine-learning-enhanced variant whose weights live in `wsaplus.pt`.

**Input and output.** Input: the single magnetogram from Task 3. Output: a 2-D map of wind speed on a `(longitude, latitude)` grid - `speed_kms` over `phi_grid_deg` x `theta_grid_deg`. This is still a *full-Sun* map; it is not yet the 1-D HUXt boundary.

**Checkpoint.** The `wsaplus.pt` weights (~317 MB) are not shipped; the cell **downloads them from Zenodo on first use** (via `scripts/fetch_wsaplus_checkpoint.py`, DOI 10.5281/zenodo.16883042). This is the one genuinely heavy compute step (neural-network inference), so the result is reusable as `wsaplus_speed_map_<event>.npz`.

> **What to look for:** broad fast-wind (>600 km/s) patches over coronal holes, and a slow-wind band (~300-400 km/s) following the magnetic neutral line. Those structures are what the CME will run into.

In [ ]:
checkpoint_path = DATA_ROOT / "wsaplus.pt"
wsaplus_cache = event_dir / f"wsaplus_speed_map_{event}.npz"

# The WSA+ checkpoint (~317 MB) is not shipped; fetch it from Zenodo on first use.
if not checkpoint_path.exists() and not wsaplus_cache.exists():
    import fetch_wsaplus_checkpoint as fw
    fw.download(checkpoint_path)
print("checkpoint exists:", checkpoint_path.exists(), "| cache exists:", wsaplus_cache.exists())

res = None
try:
    from wsaplus import generate_wsaplus_map
    res = generate_wsaplus_map(closest_map, mag_type="GONG", checkpoint_path=str(checkpoint_path))
    print("Ran WSA+ live.")
except Exception as exc:
    if wsaplus_cache.exists():
        print("WSA+ live run unavailable (", type(exc).__name__, "); loading cached map.")
        res = np.load(wsaplus_cache, allow_pickle=True)["speed_map"].item()
    else:
        raise RuntimeError(
            f"WSA+ could not run ({type(exc).__name__}: {exc}).\n"
            f"The checkpoint is data_dir/sw/wsaplus.pt; fetch it with "
            f"`python scripts/fetch_wsaplus_checkpoint.py`."
        ) from exc

print("speed_kms:", res.speed_kms.shape, float(np.nanmin(res.speed_kms)), float(np.nanmax(res.speed_kms)))
print("phi_grid_deg:", res.phi_grid_deg.shape)
print("theta_grid_deg:", res.theta_grid_deg.shape)

plt.figure(figsize=(8, 3.5))
plt.pcolormesh(res.phi_grid_deg, res.theta_grid_deg, res.speed_kms, shading="auto", cmap="viridis")
plt.colorbar(label="v [km/s]")
plt.xlabel("Carrington longitude [deg]")
plt.ylabel("latitude [deg]")
plt.title(f"{event}: WSA+ speed map")
plt.tight_layout()

## Task 5: Compute The Sub-Earth Carrington Track

*Mirrors `compute_subearth_track()`.*

HUXt is run in the ecliptic plane, so the boundary it needs is the wind speed **along the path Earth occupies** as the Sun rotates beneath it - not the whole 2-D map. The *sub-Earth point* is the spot on the Sun directly below Earth; over one Carrington rotation it sweeps through all 360 deg of longitude and wobbles a little in latitude.

**What this task computes.** For the selected rotation it steps Earth hour-by-hour from the CR start to CR+1, transforms Earth's position into Heliographic Carrington coordinates, and records `(longitude, latitude)` at each step - the curve we will sample the speed map along.

**Why latitude matters.** Earth's heliographic latitude (the B0 angle) drifts roughly +/-7.25 deg over the year. That offset means the sub-Earth track is *not* simply the map's equator - sampling the true track is what makes the boundary specific to this event's date.

> **Side note.** Carrington longitude of the sub-Earth point *decreases* with time, because the Sun rotates eastward under a (more slowly moving) Earth. Don't be surprised that the track runs "backwards" in longitude.

In [ ]:
import sunpy.coordinates.sun
from sunpy.coordinates import frames, ephemeris

t_start = sunpy.coordinates.sun.carrington_rotation_time(cr_num)
t_end = sunpy.coordinates.sun.carrington_rotation_time(cr_num + 1)
dt = t_end - t_start
n_hr = int(dt.value * 24)
obs_time = t_start + dt * np.linspace(1e-6, 1 - 1e-6, n_hr, endpoint=False)

# Vectorized: one get_earth + one transform for the whole time array (the per-sample
# loop is hundreds of calls and is very slow on Colab, where each call hits the network).
coords = ephemeris.get_earth(time=obs_time).transform_to(
    frames.HeliographicCarrington(observer="earth")
)
SBElon = np.asarray(coords.lon.value, dtype=float)
SBElat = np.asarray(coords.lat.value, dtype=float)

print(f"CR {cr_num:.2f}: {t_start.isot} -> {t_end.isot} | {n_hr} hourly samples")
print("lon range:", float(SBElon.min()), float(SBElon.max()))
print("lat range:", float(SBElat.min()), float(SBElat.max()))

## Task 6: Sample The Map Along The Track

*Mirrors `sample_interpolated()` and `sample_nearest()`.*

Now we read the WSA+ speed off the 2-D map at every `(lon, lat)` on the sub-Earth track, using two methods:

- **Interpolated** (`RegularGridInterpolator`): blends the four surrounding grid cells, giving a smooth speed series.
- **Nearest grid cell**: just takes the value of the closest cell - what the map literally says there.

The script keeps both for comparison but writes the **nearest-grid** version as the production boundary, because it preserves the map's native values without interpolation artefacts at sharp coronal-hole edges.

**The two panels:** left shows the track laid over the speed map (red = exact track, black = the nearest cells actually used); right shows the speed sampled along the track by each method.

> **What to look for:** the two curves should sit almost on top of each other, separating only where the map has steep gradients (fast/slow wind boundaries). Big separations flag places where the 1-D boundary is sensitive to the sampling choice.

In [ ]:
from scipy.interpolate import RegularGridInterpolator

speed_map = np.asarray(res.speed_kms)
lon_vals = np.asarray(res.phi_grid_deg[:, 0])
lat_vals = np.asarray(res.theta_grid_deg[0, :])

interp = RegularGridInterpolator((lon_vals, lat_vals), speed_map, bounds_error=False, fill_value=np.nan)
speed_interp = interp(np.column_stack([SBElon, SBElat]))

lon_nearest, lat_nearest, speed_nearest = [], [], []
for lon, lat in zip(SBElon, SBElat):
    i_lon = int(np.argmin(np.abs(lon_vals - lon)))
    i_lat = int(np.argmin(np.abs(lat_vals - lat)))
    lon_nearest.append(lon_vals[i_lon])
    lat_nearest.append(lat_vals[i_lat])
    speed_nearest.append(speed_map[i_lon, i_lat])
lon_nearest = np.asarray(lon_nearest)
lat_nearest = np.asarray(lat_nearest)
speed_nearest = np.asarray(speed_nearest)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
mesh = axes[0].pcolormesh(lon_vals, lat_vals, speed_map.T, shading="auto")
axes[0].scatter(SBElon, SBElat, s=8, c="red", label="sub-Earth track")
axes[0].scatter(lon_nearest, lat_nearest, s=5, c="black", alpha=0.5, label="nearest cells")
axes[0].set_xlabel("Carrington longitude [deg]")
axes[0].set_ylabel("latitude [deg]")
axes[0].set_title("Track on WSA+ map")
axes[0].legend(loc="upper right")
fig.colorbar(mesh, ax=axes[0], label="v [km/s]")

axes[1].plot(speed_interp, label="interpolated")
axes[1].plot(speed_nearest, label="nearest", alpha=0.8)
axes[1].set_xlabel("hourly sample index")
axes[1].set_ylabel("v [km/s]")
axes[1].set_title("Speed sampled along the track")
axes[1].legend()

## Task 7: Reduce The Track To The HUXt Boundary

*Mirrors the tail of `map_input_huxt()`.*

HUXt's inner boundary is a 1-D array: one wind speed per Carrington longitude degree (360 values). The track samples from Task 6 are in time order, so we **sort them by longitude** and resample onto `np.arange(1, 361)`. That array is exactly what HUXt advects outward.

**This cell writes the boundary** to `v_boundary_<event>.npz` (the same file and `speed_map` key that `generate_huxt_input.py` produces), so the GP workflow - notebook 04's live HUXt run and `gp_huxt_surrogate.py` - can load it. If a boundary already exists, it is compared before being overwritten (a re-run with the same WSA+ map reproduces it to the last digit).

> **What to look for:** the interpolated and nearest profiles should track closely, and the cell should report that it wrote a 360-point `v_boundary_<event>.npz`.

In [ ]:
lon_grid_360 = np.arange(1, 361)

def reduce_to_boundary(sample_lon, sample_speed):
    order = np.argsort(sample_lon)
    return np.interp(lon_grid_360, np.asarray(sample_lon)[order], np.asarray(sample_speed)[order])

speed_nearest_360 = reduce_to_boundary(lon_nearest, speed_nearest)
speed_interp_360 = reduce_to_boundary(SBElon, speed_interp)
boundary = speed_nearest_360   # the production (nearest-grid) HUXt inner boundary

plt.figure(figsize=(9, 3.5))
plt.plot(lon_grid_360, speed_interp_360, label="interpolated -> 360")
plt.plot(lon_grid_360, speed_nearest_360, label="nearest -> 360 (production)")

boundary_file = event_dir / f"v_boundary_{event}.npz"
# If a boundary already exists (from a previous run or generate_huxt_input.py), compare first.
if boundary_file.exists():
    prev = np.load(boundary_file)["speed_map"]
    plt.plot(lon_grid_360, prev, "--", label="existing v_boundary", alpha=0.8)
    print("existing v_boundary vs freshly computed, max|delta|:",
          float(np.nanmax(np.abs(boundary - prev))))

# Persist the boundary (same file/format as generate_huxt_input.py) so the GP workflow -
# notebook 04's live HUXt run and gp_huxt_surrogate.py - can load it.
np.savez(boundary_file, speed_map=boundary)
print(f"wrote {boundary_file} ({boundary.shape[0]} points)")

plt.xlabel("Carrington longitude [deg]")
plt.ylabel("v [km/s]")
plt.title(f"{event}: HUXt inner-boundary speed profile")
plt.legend()
plt.tight_layout()

## Task 8: Injection Time And The Seed Config

*Mirrors the injection-time block of `process_event()` and `write_event_config()`.*

With the background built, we add the CME. Four of its five parameters - `longitude`, `latitude`, `width`, `speed` - come straight from the CSV row. The fifth, `inject_hour`, is *computed*: it is how long after the model's start epoch the CME should be launched, so that it crosses 0.1 AU at the observed `cme_0p1_au` time:

```text
inject_hour = Time(cme_0p1_au) - closest_magnetogram_time   (in hours)
```

`write_event_config()` then writes `initial_theta = [inject_hour, longitude, latitude, width, speed]` plus the `cr_num` into `event_config.yaml`. That file is the **seed** the GP-surrogate workflow starts from (it is read back in notebook 04's Task 1). This cell rebuilds it, **writes it** (so the GP workflow has the seed even on a fresh checkout), and diffs it against any pre-existing copy.

> **What to look for:** the reconstructed `initial_theta` and `cr_num` should match the saved config row-for-row. A mismatch in `inject_hour` usually means a different magnetogram was selected in Task 3.

In [ ]:
inject_time = Time(selected_row["cme_0p1_au"]) - closest_time
inject_hour = inject_time.to(u.hour).value
print("cme_0p1_au:", selected_row["cme_0p1_au"], "| closest mag time:", closest_time.isot)
print("inject_hour:", inject_hour)

initial_theta = [
    inject_hour,
    selected_row["longitude"],
    selected_row["latitude"],
    selected_row["width"],
    selected_row["speed"],
]
config = {
    "initial_theta": list(map(float, initial_theta)),
    "cr_num": float(cr_num),
}

# Write the seed config (same file write_event_config() produces) so the GP workflow
# (notebook 04, gp_huxt_surrogate.py) has it; compare against any pre-existing one.
config_file = event_dir / "event_config.yaml"
if config_file.exists():
    saved_cfg = yaml.safe_load(config_file.open())
    display(pd.DataFrame({
        "parameter": ["inject_hour", "longitude", "latitude", "width", "speed"],
        "reconstructed": config["initial_theta"],
        "saved_config": saved_cfg["initial_theta"],
    }))
    print("cr_num reconstructed:", float(cr_num), "| saved:", saved_cfg["cr_num"])
else:
    display(pd.DataFrame({
        "parameter": ["inject_hour", "longitude", "latitude", "width", "speed"],
        "value": config["initial_theta"],
    }))
    print("No saved event_config.yaml yet (fresh checkout); cr_num =", float(cr_num))

with config_file.open("w") as stream:
    yaml.safe_dump(config, stream, sort_keys=False)
print("wrote", config_file)

## Task 9: Feed The Boundary To HUXt

*Mirrors the `HUXT_KWARGS` / `rhf.run_huxt_sim()` block of `process_event()`.*

This is the hand-off. The 360-point boundary becomes HUXt's `v_boundary`; together with `cr_num`, the frame, and the simulation length it defines a runnable model, and the seed `theta` defines the Cone-CME to inject into it. We assemble those kwargs here so the connection between *input preparation* (this notebook) and *running HUXt* (notebook 04) is explicit.

**The kwargs, annotated:**

| key | role |
| --- | --- |
| `v_boundary` | the 360-point inner-boundary speed profile from Task 7 |
| `cr_num` | Carrington rotation, sets the rotating-frame phase |
| `latitude=0 deg` | run in the ecliptic plane |
| `frame="sidereal"` | rotation frame for the inner boundary |
| `simtime=10 day` | how long to propagate (long enough to reach 1 AU) |
| `dt_scale=4` | output cadence (coarser than the internal time step) |

The actual seed HUXt run + arrival detection is the **live example in notebook 04 (Task 2)**, so we don't repeat it here.

In [ ]:
vboundary = boundary * (u.km / u.s)   # the live nearest-360 boundary from Task 7
HUXT_KWARGS = dict(
    v_boundary=vboundary,
    latitude=0 * u.deg,
    cr_num=cr_num,
    frame="sidereal",
    simtime=10 * u.day,
    dt_scale=4,
)
print("HUXT_KWARGS ready; boundary length:", len(vboundary))
print("seed theta:", tuple(config["initial_theta"]))
print("See notebook 04, Task 2, for a live HUXt run on this boundary.")

## Useful Commands

Everything above runs as a single script from a terminal. The flags control the two expensive/destructive bits (network download and config overwrite):

```bash
# one event, reusing local GONG FITS (no network), skipping the seed run:
python scripts/generate_huxt_input.py --event 2017-09-06 --no-download --skip-sanity-plot

# rewrite only the seed config after editing data_dir/events.csv:
python scripts/generate_huxt_input.py --event 2017-09-06 --no-download --force-config

# prepare every enabled event:
python scripts/generate_huxt_input.py --event all
```

> **Heads-up.** Without `--no-download` the script will hit the network for GONG data, and without `--skip-sanity-plot` it will launch a full seed HUXt run per event. Start with the guarded form above.

In [ ]:
RUN_EXPENSIVE = False

if RUN_EXPENSIVE:
    # Launches the full input-generation workflow (downloads + WSA+ + sanity run).
    import subprocess
    subprocess.run(
        ["python", str(BASE_DIR / "scripts" / "generate_huxt_input.py"),
         "--event", event, "--no-download", "--skip-sanity-plot"],
        check=True,
    )
else:
    print("Skipping full workflow. Set RUN_EXPENSIVE = True to run generate_huxt_input.py end to end.")